# Session 2 — Solutions

Worked solutions for object selection and b-tagging exercises. Run Session 1 solutions first, or run the cell below to load events from the 2017 config.

## Full selection and plots (Ex 2.1–2.6)


In [ ]:
# If you did not run Session 1 solutions: load events from 2017 config
if 'events' not in dir() or events is None:
    from coffea.nanoevents import NanoEventsFactory, BaseSchema
    def load_events(filepath):
        return NanoEventsFactory.from_root(filepath, schemaclass=BaseSchema).events()
    try:
        from config.datasets_2017 import get_one_file_per_group
        filesets = get_one_file_per_group()
        if filesets and "background" in filesets and filesets["background"]:
            events = load_events(filesets["background"][0])
        elif filesets and "data" in filesets and filesets["data"]:
            events = load_events(filesets["data"][0])
        else:
            raise RuntimeError("No files from config. Run Session 1 solutions or set filepath.")
    except Exception as e:
        raise RuntimeError("Load events first (Session 1 or config): " + str(e))
print("Number of events:", len(events))

In [ ]:
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt

JET_PT_MIN = 30.0
JET_ETA_MAX = 2.4
BTAG_WP_MEDIUM = 0.2783
LEP_PT_MIN = 10.0
LEP_ETA_MAX_EL = 2.5
LEP_ETA_MAX_MU = 2.4

def select_good_jets(events):
    jets = events.Jet
    mask = (jets.pt > JET_PT_MIN) & (np.abs(jets.eta) < JET_ETA_MAX) & (jets.jetId >= 2)
    return jets[mask]

def count_bjets(jets, wp=BTAG_WP_MEDIUM):
    return ak.sum(jets.btagDeepFlavB > wp, axis=1)

def select_tight_electrons(events):
    ele = events.Electron
    mask = (ele.pt > LEP_PT_MIN) & (np.abs(ele.eta) < LEP_ETA_MAX_EL) & (ele.cutBased >= 2)
    return ele[mask]

def select_tight_muons(events):
    mu = events.Muon
    mask = (mu.pt > LEP_PT_MIN) & (np.abs(mu.eta) < LEP_ETA_MAX_MU) & (mu.tightId) & (mu.pfRelIso04_all < 0.15)
    return mu[mask]

def n_tight_leptons(events):
    nele = ak.count(select_tight_electrons(events).pt, axis=1)
    nmu = ak.count(select_tight_muons(events).pt, axis=1)
    return nele + nmu

# Load events (set filepath)
# from coffea.nanoevents import NanoEventsFactory, BaseSchema
# events = NanoEventsFactory.from_root(filepath, schemaclass=BaseSchema).events()

good_jets = select_good_jets(events)
njets = ak.num(good_jets)
nbjets = count_bjets(good_jets)
nlep = n_tight_leptons(events)

print("Mean jets/event:", ak.mean(njets))
print("Events with >=2 b-jets:", ak.sum(nbjets >= 2))
print("Events with 0 leptons:", ak.sum(nlep == 0))

In [ ]:
# Jet multiplicity (Ex 2.4)
plt.figure(figsize=(6, 4))
plt.hist(ak.to_numpy(njets), bins=15, range=(0, 15), edgecolor="black", alpha=0.7)
plt.xlabel("Jet multiplicity")
plt.ylabel("Events")
plt.title("Jet multiplicity (good jets)")
plt.show()

In [ ]:
# b-jet multiplicity (Ex 2.2, 2.5)
plt.figure(figsize=(6, 4))
plt.hist(ak.to_numpy(nbjets), bins=range(7), edgecolor="black", alpha=0.7)
plt.xlabel("b-jet multiplicity")
plt.ylabel("Events")
plt.title("b-jet multiplicity (medium WP)")
plt.show()

In [ ]:
# Leading jet pT (Ex 2.6)
lead_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
plt.figure(figsize=(6, 4))
plt.hist(ak.to_numpy(lead_pt), bins=50, range=(0, 500), edgecolor="black", alpha=0.7)
plt.xlabel("Leading jet p$_T$ [GeV]")
plt.ylabel("Events")
plt.title("Leading jet pT")
plt.show()